# Fase 6 — Evaluación final y comparación de modelos

**TP1 — Aprendizaje Automático (72.75) — ITBA** · Consigna **5**

Paso **11** del pipeline de la Clase 3: selección del modelo y evaluación final en test.

**Este es el único notebook que abre el conjunto de test.** El modelo ya quedó elegido en la
Fase 5 usando exclusivamente el error de validación; acá el test sirve para una sola cosa:
estimar cómo va a funcionar ese modelo con datos nuevos.

In [1]:
# ---------------------------------------------------------------------------
# Configuracion del entorno
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

from src.config import TARGET, RANDOM_SEED
from src.data import cargar_splits
from src.preprocesamiento import separar_X_y
from src.modelado import crear_modelo_lineal, crear_modelo_polinomico, contar_features

pd.set_option("display.width", 120)

train, test = cargar_splits()
X_train, y_train = separar_X_y(train)
X_test, y_test = separar_X_y(test)

print(f"Train: {len(X_train)} filas  (se usa completo para entrenar)")
print(f"Test : {len(X_test)} filas  (se usa una sola vez, aca)")

Train: 1069 filas  (se usa completo para entrenar)
Test : 268 filas  (se usa una sola vez, aca)


---

# 1. Entrenamiento final

La Clase 2 (slide 88) indica que, una vez elegido el modelo, *"se entrena con TODO el
train+dev y ese modelo se entrega/implementa y se evalúa en test"*. Como nuestra validación
fue k-fold sobre el train, eso significa reentrenar con **las 1069 filas completas**, no con
las ~855 de cada fold.

Entrenamos dos modelos: el **elegido** (polinómica de grado 2 con λ = 100) y la **regresión
lineal** de la Fase 4, que sirve de referencia para la comparación que pide la consigna.

> Evaluar los dos en test no compromete la metodología: **la decisión ya está tomada** y
> ninguna elección depende de estos números. Lo que la Clase 2 (slide 89) prohíbe es usar el
> test *"para tomar decisiones"*.

In [2]:
# Los dos modelos a comparar. El elegido en la Fase 5 va primero.
modelos = {
    "Polinomica grado 2, lambda=100": crear_modelo_polinomico(2, alpha=100),
    "Regresion lineal": crear_modelo_lineal(),
}

# Entrenamos cada uno con el train completo.
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    print(f"{nombre}: entrenado con {len(X_train)} filas")

Polinomica grado 2, lambda=100: entrenado con 1069 filas
Regresion lineal: entrenado con 1069 filas


---

# 2. Evaluación en test

Se calcula el RMSE (métrica principal del enunciado) más MAE y R² como complementarias.

In [3]:
# Predecimos sobre el test y medimos. Es la unica vez que se usa y_test.
filas = []
for nombre, modelo in modelos.items():
    prediccion = modelo.predict(X_test)
    filas.append({
        "modelo": nombre,
        "rmse_test": round(root_mean_squared_error(y_test, prediccion), 2),
        "mae_test": round(mean_absolute_error(y_test, prediccion), 2),
        "r2_test": round(r2_score(y_test, prediccion), 3),
    })

resultados_test = pd.DataFrame(filas).set_index("modelo")
resultados_test

,rmse_test,mae_test,r2_test
modelo,,,
"Polinomica grado 2, lambda=100",4630.26,2879.09,0.883
Regresion lineal,5956.34,4177.05,0.807


---

# 3. Validación contra test

Comparamos lo que estimamos en la Fase 5 con lo que efectivamente dio el test.

| Modelo | RMSE validación | RMSE test | Diferencia |
|---|---|---|---|
| Polinómica g2, λ = 100 | 4.905,34 | **4.630,26** | −275,08 |
| Regresión lineal | 6.123,65 | **5.956,34** | −167,31 |

**Los dos modelos dieron mejor en test que en validación.** Conviene explicar por qué, porque
lo esperable suele ser lo contrario:

1. **El modelo final se entrena con más datos.** En la validación cruzada cada modelo se
   ajustaba con ~855 filas; el final usa las 1069. Más datos, mejor ajuste.
2. **El test es una muestra de 268 filas y tiene su propia variabilidad.** El intervalo que
   calculamos abajo lo cuantifica.

Lo importante es que **el orden entre los modelos se mantiene**: el polinómico gana en
validación y gana en test. La validación cruzada eligió bien.

---

# 4. ¿Cuánta confianza tiene el RMSE de test?

El RMSE de test es **un número calculado sobre 268 observaciones**, no una constante. Para
saber cuánto podría haber variado, remuestreamos los residuos con bootstrap: se sortean 268
residuos con reposición, se recalcula el RMSE, y se repite 5000 veces.

In [4]:
# Bootstrap sobre los residuos del modelo elegido.
modelo_elegido = modelos["Polinomica grado 2, lambda=100"]
residuos = y_test.to_numpy() - modelo_elegido.predict(X_test)

rng = np.random.default_rng(RANDOM_SEED)
rmse_bootstrap = [
    np.sqrt(np.mean(rng.choice(residuos, size=len(residuos), replace=True) ** 2))
    for _ in range(5000)
]

rmse_test = root_mean_squared_error(y_test, modelo_elegido.predict(X_test))
inferior, superior = np.percentile(rmse_bootstrap, [2.5, 97.5])

print(f"RMSE de test         : {rmse_test:>10,.2f}")
print(f"Intervalo 95%        : [{inferior:,.0f} , {superior:,.0f}]")
print(f"Amplitud             : {superior - inferior:,.0f}")
print(f"\nRMSE como % del costo medio del test ({y_test.mean():,.0f}): "
      f"{100 * rmse_test / y_test.mean():.1f}%")

RMSE de test         :   4,630.26
Intervalo 95%        : [3,814 , 5,403]
Amplitud             : 1,589

RMSE como % del costo medio del test (14,272): 32.4%


El intervalo va de **3.814 a 5.403**: una amplitud de casi 1.600 dólares. Contiene sin
problema el 4.905 que había estimado la validación cruzada, lo que confirma que ambas
mediciones son compatibles entre sí.

---

# 5. Las tres preguntas de la consigna 5

## 5.1 ¿Qué modelo obtuvo menor error?

**La regresión polinómica de grado 2 con regularización L1 (λ = 100).**

| | Regresión lineal | Polinómica g2, λ = 100 |
|---|---|---|
| RMSE validación | 6.123,65 | **4.905,34** |
| RMSE test | 5.956,34 | **4.630,26** |
| MAE test | 4.177,05 | **2.879,09** |
| R² test | 0,810 | **0,880** |

Gana en las cuatro métricas. En test reduce el RMSE un **22,3 %** respecto del lineal.

La mejora no es casual: el modelo lineal tenía un gap de sólo 47 dólares entre train y
validación, o sea que **no estaba sobreajustando sino quedándose corto**. El diagnóstico de
la Fase 4 fue *underfitting*, y la respuesta correcta era darle más capacidad.

**Un matiz honesto:** en la Fase 5 vimos que el grado 3 con λ = 100 es *estadísticamente
equivalente* al grado 2 (6 dólares de diferencia sobre 50 particiones, con un desvío de 382).
La afirmación defendible es que **ambos modelos con λ = 100 son mejores que el lineal**, no
que el grado 2 sea mejor que el grado 3.

## 5.2 ¿Cuál implementarían en una aplicación real?

**El mismo: polinómica de grado 2 con λ = 100.** Tres razones.

**1. La diferencia de error es grande en términos de negocio.** 1.326 dólares menos de RMSE
por póliza no es un detalle estadístico: es dinero que la aseguradora deja de errar al
estimar el costo de un asegurado.

**2. El costo de implementación es el mismo.** Todo el modelo —encoding, escalado, expansión
polinómica y regresión— vive dentro de un único objeto `Pipeline`. Poner en producción el
polinómico es exactamente el mismo trabajo que poner el lineal: se llama a `.predict()` con
las 6 variables originales y el objeto se encarga del resto.

**3. Sigue siendo un modelo chico.** La penalización L1 dejó **20 features activas de 44**.
No es una caja negra: es una regresión lineal sobre 20 términos.

**Qué se pierde, y por qué lo aceptamos.** El lineal es más fácil de explicar: *"fumar suma
23.078 dólares al año"* es una frase que cualquiera entiende. En el polinómico ese efecto se
reparte entre `smoker_yes` y sus términos cruzados, y ya no hay un número único que citar.

Es una pérdida real, pero manejable: para explicarle a un cliente por qué se le cotiza un
precio, se puede mostrar cuánto cambia la predicción al modificar una variable, sin necesidad
de leer los coeficientes uno por uno.

**Si el requisito fuera regulatorio** —por ejemplo, tener que justificar cada término del
precio ante un organismo de control— la respuesta cambiaría y elegiríamos el lineal,
aceptando el 22 % de error adicional.

## 5.3 ¿Qué RMSE esperar cuando el modelo se use?

**Diríamos que esperamos un RMSE del orden de 4.900 dólares, no de 4.630.**

Puede sonar raro prometer un número peor que el que dio el test, pero es lo correcto:

**Por qué no prometemos el 4.630 del test.** Es una medición sobre 268 observaciones, con un
intervalo de confianza del 95 % que va de **3.814 a 5.403**. Elegir el extremo optimista de
esa banda sería vender una precisión que no está garantizada.

**Por qué 4.900.** Es la estimación de la validación cruzada, que promedia **50 particiones
distintas** en lugar de una sola. Es la estimación más estable que tenemos, y además es
conservadora: queda por encima de lo que efectivamente midió el test.

**La respuesta completa que daríamos:**

> *"Esperamos un RMSE en torno a los 4.900 dólares, que es aproximadamente un tercio del costo
> medio anual. En la evaluación final sobre datos nunca vistos obtuvimos 4.630, con un
> intervalo de confianza del 95 % entre 3.814 y 5.403."*

**Y dos condiciones que hay que declarar:**

**a) El modelo vale para la población que vio.** El train cubre edades de **18 a 64 años** y
BMI de **15,96 a 53,13**. Para una persona de 80 años el modelo devuelve un número igual, pero
es una extrapolación sin respaldo empírico: nunca vio un caso así.

**b) El error no se reparte de forma pareja.** Como vimos desde la Fase 2, `charges` son dos
poblaciones. El modelo acierta mucho mejor en los costos bajos que en los altos, así que en el
segmento de fumadores el error real va a ser mayor que ese promedio de 4.900.

Que el **MAE sea 2.879** mientras el **RMSE es 4.630** confirma esa asimetría: el RMSE eleva
los errores al cuadrado, así que la brecha entre ambos indica que unos pocos casos concentran
buena parte del error total.

---

## Conclusiones de la Fase 6

| | Valor |
|---|---|
| Modelo entregado | Polinómica grado 2 + Lasso (λ = 100), entrenado con las 1069 filas de train |
| Features activas | 20 de 44 |
| **RMSE en test** | **4.630,26** |
| Intervalo 95 % | [3.814 , 5.403] |
| MAE en test | 2.879,09 |
| R² en test | 0,880 |
| RMSE que se promete | ~4.900 (estimación de validación cruzada, más conservadora) |
| Rango de validez | `age` 18–64 · `bmi` 15,96–53,13 |

**El test se usó una sola vez y no volvió a tocarse.**

Con esto quedan cubiertas todas las consignas del enunciado: 1.1 a 1.4, 2.1 a 2.3, 3.1 a 3.3,
4 y 5.